This Notebook preprocesses the .csv file of friends for the text mining course

Step 1: Importing CSV file

In [2]:
import pandas as pd 

try:
    df = pd.read_csv('Friends.csv')
    df = df.astype("string")
except FileNotFoundError:
    print("File not found.")
except Exception as e:
    print(f"An error occurred: {e}")
    
print("File read successfully.")
print("Headers")
print(df.head())

File read successfully.
Headers
                                                Text  Speaker  \
0  Originally written by Marta Kauffman and David...     <NA>   
1                          Transcribed by guineapig.     <NA>   
2   CENTRAL PERK. (ALL PRESENT EXCEPT RACHEL AND ...  SCENE 1   
3   There's nothing to tell! He's just some guy I...   MONICA   
4   C'mon, you're going out with the guy! There's...     JOEY   

                                             Episode     Season     Show  
0  Episode-01-The One Where Monica Gets a New Roo...  Season-01  Friends  
1  Episode-01-The One Where Monica Gets a New Roo...  Season-01  Friends  
2  Episode-01-The One Where Monica Gets a New Roo...  Season-01  Friends  
3  Episode-01-The One Where Monica Gets a New Roo...  Season-01  Friends  
4  Episode-01-The One Where Monica Gets a New Roo...  Season-01  Friends  


Step2: remove punctuation and special characters
https://stackoverflow.com/questions/39782418/remove-punctuations-in-pandas

In [3]:
import string
df_no_punct = df.copy()
extra_characters = 'ˆ‘"”∼'
characters_to_remove = string.punctuation + extra_characters
for col in df.columns:
    df_no_punct[col] = df_no_punct[col].str.translate(str.maketrans("", "", characters_to_remove))
# Test case
test = True
if test == True:
    text = 'Hello!”#$%&()*+,-.:;?@[]\\ˆ ‘{|}∼World. We are testing the preprocessing of the text of the TV show Friends.'
    text_no_punct = text.translate(str.maketrans("", "", characters_to_remove))
    print(text_no_punct)

Hello World We are testing the preprocessing of the text of the TV show Friends


Step 3: Lower case all the text
https://stackoverflow.com/questions/22245171/how-to-lowercase-a-pandas-dataframe-string-column-if-it-has-missing-values

In [4]:
df_lower = df_no_punct.copy()
for col in df_lower.columns:
    df_lower[col] = df_lower[col].str.lower()
    
#Test case 
if test == True:
    text_lower = text_no_punct.lower()
    print(text_lower)   


hello world we are testing the preprocessing of the text of the tv show friends


Step 4: tokenize the text 
https://www.geeksforgeeks.org/nlp/tokenize-text-using-nltk-python/

In [5]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize, word_tokenize

col = "Text"

df_tokenized_sent = df_lower.copy()
df_tokenized_word = df_lower.copy()

df_tokenized_sent[col] = df_lower[col].dropna().apply(sent_tokenize)
df_tokenized_word[col] = df_lower[col].dropna().apply(word_tokenize)

#Test case
if test == True:
    text_sent_tokenized = sent_tokenize(text_lower)
    text_word_tokenized = word_tokenize(text_lower)
    print(text_sent_tokenized)
    print(text_word_tokenized)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Louis\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Louis\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


['hello world we are testing the preprocessing of the text of the tv show friends']
['hello', 'world', 'we', 'are', 'testing', 'the', 'preprocessing', 'of', 'the', 'text', 'of', 'the', 'tv', 'show', 'friends']


Step 5: remove stopwords
https://www.geeksforgeeks.org/nlp/removing-stop-words-nltk-python/

In [6]:
from nltk.corpus import stopwords
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

df_tokenized_word[col] = df_tokenized_word[col].dropna().apply(
        lambda tokens: [token for token in tokens if token not in stop_words]
    )

#Test case
if test == True:
    text_no_stopwords = [token for token in text_word_tokenized if token not in stop_words]
    print(text_no_stopwords)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Louis\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


['hello', 'world', 'testing', 'preprocessing', 'text', 'tv', 'show', 'friends']


Step 5: Use porter stemmer to stem the words
https://www.geeksforgeeks.org/nlp/porter-stemmer-technique-in-natural-language-processing/

In [7]:
from nltk.stem import PorterStemmer
porter_stemmer = PorterStemmer()

df_tokenized_word[col] = df_tokenized_word[col].dropna().apply(
        lambda tokens: [porter_stemmer.stem(token) for token in tokens]
    )

print(df_tokenized_word.head())

                                                Text  Speaker  \
0   [origin, written, marta, kauffman, david, crane]     <NA>   
1                             [transcrib, guineapig]     <NA>   
2     [central, perk, present, except, rachel, ross]  scene 1   
3                 [there, noth, tell, he, guy, work]   monica   
4  [cmon, your, go, guy, there, got, ta, someth, ...     joey   

                                             Episode    Season     Show  
0  episode01the one where monica gets a new roomm...  season01  friends  
1  episode01the one where monica gets a new roomm...  season01  friends  
2  episode01the one where monica gets a new roomm...  season01  friends  
3  episode01the one where monica gets a new roomm...  season01  friends  
4  episode01the one where monica gets a new roomm...  season01  friends  


Step 6: Perform POS tagging and NER 
https://spacy.io/usage/linguistic-features

In [ ]:
import spacy
from tqdm.auto import tqdm

nlp = spacy.load("en_core_web_sm")
results = []
text_rows = df_lower[col].dropna()

for row_index, row in tqdm(text_rows.items(), total=len(text_rows)):
    doc = nlp(row)
    
    for token in doc:
        results.append({
            "row_index": row_index,
            "text": token.text,
            "lemma": token.lemma_,
            "pos": token.pos_,
            "tag": token.tag_,
            "dependency": token.dep_,
            "shape": token.shape_,
            "is_alpha": token.is_alpha,
            "is_stop": token.is_stop,
            "ent_type": token.ent_type_,
            "ent_iob": token.ent_iob_
        })
        
print(results[:5])

print("Data preprocessing completed successfully.")


  0%|          | 0/69974 [00:00<?, ?it/s]